In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

schema=StructType([
    StructField('Pregnancies', IntegerType(), True),
    StructField('Glucose', IntegerType(), True),
    StructField('BloodPressure', IntegerType(), True),
    StructField('SkinThickness', IntegerType(), True),
    StructField('DiabetesPedigreeFunction', FloatType()),
    StructField('Age',IntegerType()),
    StructField('Outcome', IntegerType())
])

df=spark.read.format('csv').options(inferSchema='true', header='true', schema=schema).load('/Volumes/dbws01/default/ml_lab/diabetes.csv').dropna()


In [0]:
splits=df.randomSplit([0.7, 0.3])
train=splits[0]
test=splits[1]
print(train.count(), test.count())

551 217


In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, recall_score, precision_score
import pandas as pd

In [0]:
with mlflow.start_run():
    numFeatures=['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
    train_pd=train.select(numFeatures+['Outcome']).toPandas()
    test_pd=test.select(numFeatures+['Outcome']).toPandas()
    x_train=train_pd[numFeatures]
    y_train=train_pd['Outcome']
    x_test=test_pd[numFeatures]
    y_test=test_pd['Outcome']
    scaler=MinMaxScaler()
    x_train_scaled=scaler.fit_transform(x_train)
    x_test_scaled=scaler.transform(x_test)
    max_iter=20
    model=LogisticRegression(max_iter=max_iter)
    model.fit(x_train_scaled, y_train)
    mlflow.log_param('max_iter', max_iter )

    y_pred=model.predict(x_test_scaled)
    accuracy=accuracy_score(y_test,y_pred)
    recall=recall_score(y_test, y_pred, average='weighted')
    mlflow.log_metric('accuracy', accuracy)
    mlflow.log_metric('recall', recall)
    
    #create signature
    input_example=pd.DataFrame([{
        'Pregnancies':8, 'Glucose':85 , 'BloodPressure':65,
        'SkinThickness':29, 'Insulin':0, 'BMI':26.6, 'DiabetesPedigreeFunction':0.351, 'Age':31
    }])
    output_example=pd.DataFrame([{
        'prediction':[1]
    }])
    signature=infer_signature(input_example, output_example)
    
    mlflow.sklearn.log_model(sk_model=model, signature=signature,
                             input_example=input_example,
                             artifact_path='diabetes_sklearn')



/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/04 14:35:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://adb-7405610296555497.17.azuredatabricks.net/ml/experiments/1775294967107574/models/m-e3772447a4964718

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import optuna

def objective(trial):
    max_depth=trial.suggest_int('Maxdepth',1,20)
    max_bins=trial.suggest_categorical('MaxBins',[10,20,30])
    with mlflow.start_run(nested=True):
        numFeatures=['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
        numvector=VectorAssembler(inputCols=numFeatures, outputCol='numericFeatures')
        scaler=MinMaxScaler(inputCol=numvector.getOutputCol(), outputCol='normalizedFeatures')
    
        mlalgo=DecisionTreeClassifier(labelCol='Outcome', featuresCol='normalizedFeatures', maxDepth=max_depth, maxBins=max_bins)
        
        pipeline=Pipeline(stages=[numvector, scaler, mlalgo])
        pipelineModel=pipeline.fit(train)

        #evaluate
        prediction=pipelineModel.transform(test)
        evaluator=MulticlassClassificationEvaluator(labelCol='Outcome', predictionCol='prediction', metricName='accuracy')
        accuracy=evaluator.evaluate(prediction)
        mlflow.log_metric('accuracy', accuracy)
        mlflow.log_param('max_depth', max_depth)
        mlflow.log_param('max_bins', max_bins)
        input_example=pd.DataFrame([{
        'Pregnancies':8, 'Glucose':85 , 'BloodPressure':65,
        'SkinThickness':29, 'Insulin':0, 'BMI':26.6, 'DiabetesPedigreeFunction':0.351, 'Age':31
    }])
        output_example=pd.DataFrame([{
        'prediction':[1]
    }])
        signature=infer_signature(train.select(numFeatures).toPandas(), prediction.select('prediction').toPandas())
        mlflow.spark.log_model(pipelineModel, 'model', signature=signature, dfs_tmpdir="/Volumes/dbws01/default/ml_lab/tmp")

        return accuracy


        
        


In [0]:
df.columns

['Pregnancies',
 'Glucose',
 'BloodPressure',
 'SkinThickness',
 'Insulin',
 'BMI',
 'DiabetesPedigreeFunction',
 'Age',
 'Outcome']

In [0]:
with mlflow.start_run(run_name='optuna_hyperparam'):
    study=optuna.create_study(direction='maximize', study_name='diabetes_classifier')
    #optimize
    study.optimize(objective, n_trials=3)
    mlflow.log_params(study.best_params)
    mlflow.log_metric('best_accuracy', study.best_value)


[I 2026-10-04 15:41:32,325] A new study created in memory with name: diabetes_classifier
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/04 15:41:54 INFO mlflow.spark: Inferring pip requirements by reloading the logged model from the databricks artifact repository, which can be t

2026/10/04 15:42:11 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: dbfs:/databricks/mlflow-tracking/1775294967107574/0004c863ea0f446dbe814e242aecf7a4/artifacts/model/sparkml, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 


Uploading artifacts:   0%|          | 0/4 [00:00<?, ?it/s]

[I 2026-10-04 15:42:12,407] Trial 0 finished with value: 0.7004608294930875 and parameters: {'Maxdepth': 2, 'MaxBins': 10}. Best is trial 0 with value: 0.7004608294930875.
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/04 15:42:32 INFO mlflow.spark: Inferring pip requirements by

2026/10/04 15:42:48 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: dbfs:/databricks/mlflow-tracking/1775294967107574/1ffe419bce2943cab639bf862c77086f/artifacts/model/sparkml, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 


Uploading artifacts:   0%|          | 0/4 [00:00<?, ?it/s]

[I 2026-10-04 15:42:49,226] Trial 1 finished with value: 0.7050691244239631 and parameters: {'Maxdepth': 12, 'MaxBins': 20}. Best is trial 1 with value: 0.7050691244239631.
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/04 15:43:09 INFO mlflow.spark: Inferring pip requirements b

2026/10/04 15:43:24 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: dbfs:/databricks/mlflow-tracking/1775294967107574/9d2d2fdaeec042828b82915ef74544f2/artifacts/model/sparkml, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 


Uploading artifacts:   0%|          | 0/4 [00:00<?, ?it/s]

[I 2026-10-04 15:43:25,024] Trial 2 finished with value: 0.7142857142857143 and parameters: {'Maxdepth': 6, 'MaxBins': 20}. Best is trial 2 with value: 0.7142857142857143.
